In [1]:
%load_ext autoreload
%autoreload 2

# Importing the modules
import os
os.environ['OMP_NUM_THREADS'] = '8' # Force the system to allocate 8 threads to OpenMP
os.environ['NGS_NUM_THREADS'] = '8' # Force NGSolve's internal task manager to use 8 threads
import ngsolve as ngs
ngs.SetNumThreads(8)

In [1]:
import config_dict as cfg                    # config = physical & simulation parameters 
from solver_2DHcurl_1DH1 import *            # solver = mesh build, physics import and FEM method + pmls initialization
# from solver_2DHcurl_1DH1_Copie import *
import PP_Run_and_Save as pp_run     # post process run and save = run scan function and save data in H5 files
import PP_Plot_and_Load as pp_plot   # post process load and plot = recover the sim data from H5 files and plot graphs "instantly"

import Antenna_Desc_n_Plot as antenna

geom_mode = "2D" # "1D" or "2D" 
box_medium = "PLASMA" # "VACUUM" or "PLASMA" 

antenna_grill = None
if antenna_grill != None:
    antenna_grill = antenna.AntennaGrill(b_active=0.009, d_septa=0.001, d_gap=0.002, Lx_wg_active=0.08, Lx_wg_passive=0.02, b_passive=0.009)
    num_active= 6
    n_modules = 3
    is_PAM=False
    for module in range(n_modules):
        antenna_grill.add_module(num_active, is_PAM=False, delta_phi_deg=-150, amplitude=200.34)
        module+=1

    instructions = antenna_grill.generate_mesh_instructions()
    antenna.plot_antenna_blueprint(instructions)

solver = LHCouplingSolver_2DHcurl_1DH1(cfg.__dict__, geom_mode, box_medium, antenna_grill)

In [8]:
# %%capture
mesh = solver.build_mesh_with_PMLs()
solver.build_physics_Stix_B_field()

GF_E_field, Gamma_R, Gamma_T, diag_data = solver.solve_helmholtz_2DHcurl_1DH1_with_pml(mesh, geom_mode, box_medium)

===== Simulation : PLASMA 2D with ANTENNA =====
==== 
Lx_plasma: 5.00e-02m,   Lx_pml: 4.37e-02m,    Lx_tot: 9.37e-02m
Lx_wg = 8.00e-02m
Lz_antenna: 2.19e-01m, Lz_wall: 2.00e-02m
Lz_plasma: 2.19e-01m, Lz_pml: 2.00e-01m, Lz_tot: 6.59e-01m
==== 
n_∥: 2.0+0.0j,   λ_∥: 4.05e-02m 
n_⟂⁺ (edge): 2.41+0.00j, n_⟂⁺ (core): 7.41
λ_⟂⁺ (core): 1.09e-02m 
n_⟂-:0.00+1.73j, λ_⟂⁻: 4.68e-02m
==== 
n_index_meshing: 7.41 
λ_meshing (shortest): 1.09e-02m 
maxh_plasma: 7.29e-04m
===== Because 2D: Radial and Toroidal PMLs =====
[DOMAIN 2D DEFINED AND GLUED]
DOMAIN MESH SET
[MESH GENERATED !]
Lx_wg: 8.00e-02m, wg_medium: VACUUM
Waveguide medium is not PLASMA, wg_is_plasma set to 0.0
==== 
#DoFs = 12113663
len(instruction): 43
n_perp_vac_port: 1.00+0.00j


: 

In [2]:
# solver.plot_radial_density_profile()

In [5]:
# Create the Run_XXX_XXX file in Simulation_Results folder
# run_folder = /Simulation_Results/Run_XXX
run_folder_path = pp_run.setup_output_directory("Simulation_Results", save_data=True)
print(f'run_folder_path: {run_folder_path}')
# Total Run_XXX folder path to Remi's shared zone (cea intra) 
# run_folder_tot_path = "/home/remi/Perso/Stage/M2_IRFM/Codes/2D_Complete_Version/" + run_folder_path
if run_folder_path is not None:
    run_folder_tot_path = '/Home/RB286887/LH_coupling_code_remi/2D_Complete_Version/' + run_folder_path
    print(f'sim_target_folder: {run_folder_tot_path}')
else: 
    run_folder_tot_path = None


[SYSTEM] Output directory created: Simulation_Results/Run_20260728_083305
run_folder_path: Simulation_Results/Run_20260728_083305
sim_target_folder: /Home/RB286887/LH_coupling_code_remi/2D_Complete_Version/Simulation_Results/Run_20260728_083305


In [5]:
# E_map2D_h5_filepath = pp_run.run_2D_wave_map(mesh, GF_E_field, cfg, run_folder_tot_path, geom_mode, box_medium, antenna_grill, diag_data)

# Lz_wall = cfg.DOMAIN.get('Lz_wall', 0.02)

# # pp_plot.plot_2D_wave_map(E_map2D_h5_filepath, run_folder_tot_path, geom_mode, component='E_norm', 
# #                         value_type='real',antenna_grill=antenna_grill, Lz_wall=Lz_wall, plot_poynting=False, 
# #                         show_windows_R=False, show_windows_T=False, Poynting_box=False)


In [4]:
# pp_plot.plot_1D_radial_slice_with_theory(
#     h5_filepath=E_map2D_h5_filepath, 
#     cfg=cfg.__dict__, 
#     component='Ez', 
#     z_eval=0.20)  # None for exact center

In [6]:
# # # Test n_para spectrum
# # saved_mat_file = 'matlab_data_files/saved_sc_FAM_8mods_1wg.mat'
# saved_mat_file = 'matlab_data_files/saved_sc_swan_0.mat'
# n_para_array, FEM_power_spectrum = pp_plot.plot_n_para_spectrum(mesh, GF_E_field, cfg, geom_mode, 
#                                                             run_folder_tot_path, saved_mat_file, diag_data, 
#                                                             ALOHA_spec_comparison=True, x_eval=0.001)


In [7]:
# gamma_dict = solver.compute_waveguide_S_parameters()
# import os
# import numpy as np
# import matplotlib.pyplot as plt
# import h5py 

# def plot_waveguide_S_parameters(gamma_dict, saved_mat_file, figure_save_dir):
#     """
#     Extracts and plots the Active Reflection Coefficients (Power Reflectivity and Phase)
#     for a phased array antenna from a dictionary of S-parameters.
#     """
#     if not gamma_dict:
#         print("[!] No S-parameter data to plot.")
#         return

#     # 1. Secure Data Extraction and Sorting
#     # Sort the dictionary items based on their physical 'z_center' to ensure 
#     # they are plotted in the correct geometric order from left to right.
#     sorted_wgs = sorted(gamma_dict.items(), key=lambda item: item[1]['z_center'])
#     wg_labels = []
#     refl_list = [data['Power_Reflectivity']*100 for wg, data in gamma_dict.items()]
    
#     for wg_name, data in sorted_wgs:
#         # Format label (e.g., 'wg_1' -> 'WG 1')
#         wg_labels.append(wg_name.upper().replace('_', ' '))
                         
#     with h5py.File(saved_mat_file, 'r') as f:
#         Coeff_Ref_back_wg = f['scenario/results/CoeffRefPuiss'][:].flatten()
#     print('Coeff_Ref_back_wg: ', Coeff_Ref_back_wg)
#     print(f'mean Coeff Ref back wg: {np.mean(Coeff_Ref_back_wg)}')
#     # 2. Figure Initialization (Stacked Subplots)
#     plt.figure(figsize=(10, 6))
#     x_indices = np.arange(len(wg_labels))+1

#     # ==========================================
#     # Top Panel: Power Reflectivity (|Gamma|^2)
#     # ==========================================
#     plt.plot(x_indices, refl_list, color='crimson', label='FEM', linewidth=1.2, zorder=3)
#     plt.plot(x_indices, Coeff_Ref_back_wg, color='blue', label='ALOHA',linewidth=1.2, zorder=3)

#     plt.ylabel(r'Power Reflectivity $|\Gamma|^2$ [%]', fontsize=14)
#     plt.ylim(0, max(max(refl_list) * 1.25, 5.0)) # Scale y-axis dynamically
#     plt.grid(True, which='both', linestyle='--', alpha=0.6, zorder=0)
#     plt.tick_params(direction='in', length=6, width=1.5, bottom=True, top=True, left=True, right=True)
#     plt.legend(loc='best', fontsize=14)
#     # ==========================================
#     # ==========================================
#     # Global Formatting
#     # ==========================================
#     # Set the x-axis labels to the waveguide names
#     plt.xticks(x_indices, label=wg_labels, fontsize=12)
#     plt.xlabel("Active Waveguides", fontsize=16)

#     plt.tight_layout()
#     if figure_save_dir is not None:
#         os.makedirs(figure_save_dir, exist_ok=True)
#         fig_path = os.path.join(figure_save_dir, "Waveguide_S_Parameters.pdf")
#         plt.savefig(fig_path, dpi=300, bbox_inches='tight')
#         print(f"--- S-Parameter plot saved to {fig_path} ---")
#     plt.show()

# plot_waveguide_S_parameters(gamma_dict, saved_mat_file, run_folder_tot_path)

In [8]:
# # ===========================
# x_target=0.000
# aperture_fields = solver.extract_tangential_aperture_fields(x_target)
# # # ===========================

# import matplotlib.pyplot as plt
# import numpy as np
# import config_dict as cfg  

# # mat_file_path = 'matlab_data_files/saved_sc_FAM_8mods_1wg.mat'
# def plot_aperture_fields(field_data, mat_file_path, x_target, save_dir=None):
#     """
#     Plots the complex electric field components along the toroidal (z) direction.
#     Ideal for comparing the aperture fields (x=0) with ALOHA.
#     """
#     z_coords = field_data['z_coords']
#     Ez = field_data['Ez']
#     Lz_wall = cfg.DOMAIN['Lz_wall']
#     fig, (ax1, ax2) = plt.subplots(2,1, sharex=True, figsize=(6, 6))
    
#     # Plot Absolute, Real, and Imaginary parts of Ez (the primary coupled field)
#     ax1.text(0.0, 0.85*np.max(np.abs(Ez)), 'FEM', rotation=0, va='bottom', color='crimson', fontsize=16, fontweight='bold')
#     ax1.plot(z_coords, np.real(Ez), color='crimson', lw=1., linestyle='-', label=r'$Re(E_z)$')
#     # ax1.plot(z_coords, np.imag(Ez), color='royalblue', lw=1., linestyle='-', label=r'$Im(E_z)$')
#     ax1.plot(z_coords, np.abs(Ez), color='black', lw=1, label=r'$|E_z|$')

#     # Formatting
#     ax1.set_xlabel("Toroidal Position z [m]", fontsize=14)
#     ax1.set_ylabel("Electric Field E_z [V/m]", fontsize=14)
#     ax1.set_title(f"Aperture Field Profile at x = {x_target:.4f} m", fontsize=14, fontweight='bold')
    
#     ax1.tick_params(direction='in', length=6, width=1.5, bottom=True, top=True, left=True, right=True)
#     ax1.grid(True, which='both', linestyle='--', alpha=0.5)
#     ax1.legend(loc='best', framealpha=0.95, fontsize=10, ncol=1)
    

#     with h5py.File(mat_file_path, 'r') as f:
#         # 1. Extraction des données brutes
#         abs_z = f['scenario/results/abs_z'][:]          
#         E_mouth_brut = f['scenario/results/E_mouth'][:] 
        
#     z_coords = abs_z[:]
#     Ez_mouth = E_mouth_brut[:, 2]
    
#     Ez_mouth_norm = np.sqrt((Ez_mouth['real'])**2 + (Ez_mouth['imag'])**2)
#     ax2.text(0.00, 0.85*np.max(Ez_mouth_norm), 'ALOHA', rotation=0, va='bottom', color='royalblue', fontsize=16, fontweight='bold')

#     ax2.plot(z_coords+Lz_wall, Ez_mouth['real'], color='crimson', label=r'$Re(E_z)$', lw=1.) # , label=f'Rangée {rangee + 1}', linewidth=1.2)
#     # ax2.plot(z_coords+Lz_wall, Ez_mouth['imag'], color='royalblue', label=r'$Im(E_z)$', lw=1.) # , label=f'Rangée {rangee + 1}', linewidth=1.2)
#     ax2.plot(z_coords+Lz_wall, Ez_mouth_norm, color='black', label=r'$|E_z|$', linewidth=1.) # , label=f'Rangée {rangee + 1}', linewidth=1.2)
    
#     ax2.tick_params(direction='in', length=6, width=1.5, bottom=True, top=True, left=True, right=True)
#     ax2.grid(True, which='both', linestyle='--', alpha=0.5)
#     ax2.legend(loc='best', framealpha=0.95, fontsize=10, ncol=1)
    
#     plt.tight_layout()
    
#     if save_dir:
#         import os
#         filepath = os.path.join(save_dir, f"Aperture_Field_x{x_target:.4f}.pdf")
#         plt.savefig(filepath, dpi=300)
#         print(f"  -> Aperture field plot saved to {filepath}")
        
#     plt.show()
# # ===========================
# plot_aperture_fields(aperture_fields, saved_mat_file, x_target)
# # ===========================


In [17]:
import h5py
chemin_fichier = 'matlab_data_files/saved_sc_FAM_8mods_1wg.mat' 
def afficher_structure(nom, objet):
    """
    Cette fonction est appelée pour chaque élément trouvé dans le fichier HDF5.
    """
    if isinstance(objet, h5py.Group):
        # C'est une structure MATLAB (un "dossier")
        print(f"📁 Structure : {nom}")
        
    elif isinstance(objet, h5py.Dataset):
        # C'est une donnée (matrice, vecteur, etc.)
        # On récupère ses dimensions (shape) et son type de donnée (dtype)
        print(f"   ↳ 📄 Donnée : {nom} | Dimensions : {objet.shape} | Type : {objet.dtype}")

# Ouverture et exploration du fichier
print(f"--- Analyse du fichier : {chemin_fichier} ---")
with h5py.File(chemin_fichier, 'r') as f:
    f.visititems(afficher_structure)



--- Analyse du fichier : matlab_data_files/saved_sc_FAM_8mods_1wg.mat ---
📁 Structure : scenario
📁 Structure : scenario/antenna
   ↳ 📄 Donnée : scenario/antenna/a_ampl | Dimensions : (1, 8) | Type : float64
   ↳ 📄 Donnée : scenario/antenna/a_phase | Dimensions : (1, 8) | Type : float64
   ↳ 📄 Donnée : scenario/antenna/architecture | Dimensions : (13, 1) | Type : uint16
   ↳ 📄 Donnée : scenario/antenna/freq | Dimensions : (1, 1) | Type : float64
📁 Structure : scenario/antenna_lh
   ↳ 📄 Donnée : scenario/antenna_lh/archName | Dimensions : (13, 1) | Type : uint16
   ↳ 📄 Donnée : scenario/antenna_lh/beam | Dimensions : (2,) | Type : uint64
   ↳ 📄 Donnée : scenario/antenna_lh/frequency | Dimensions : (1, 1) | Type : float64
   ↳ 📄 Donnée : scenario/antenna_lh/n_par | Dimensions : (2,) | Type : uint64
   ↳ 📄 Donnée : scenario/antenna_lh/name | Dimensions : (40, 1) | Type : uint16
   ↳ 📄 Donnée : scenario/antenna_lh/plasmaedge | Dimensions : (2,) | Type : uint64
   ↳ 📄 Donnée : scenario/anten

In [9]:
import copy
# Assuming your solver class is imported as:
# from solver_2DHcurl_1DH1_3 import LHCouplingSolver_2DHcurl_1DH1

def run_ne0_parameter_scan(base_cfg, ne0_values, geom_mode, box_medium, antenna_grill, save_filename="fem_scan_results.npz"):
    """
    Loops over a list of ne0 values, runs the FEM solver, computes the average 
    waveguide reflection, and saves the results for comparison with ALOHA.
    """
    print(f"--- Starting FEM ne0 Scan ({len(ne0_values)} points) ---")
    
    avg_reflections = np.zeros(len(ne0_values))
    
    for i, ne0 in enumerate(ne0_values):
        print(f"\n[{i+1}/{len(ne0_values)}] Solving for ne0 = {ne0:.2e} m-3")
        
        # 1. Safely copy and update the configuration
        # We use deepcopy to ensure we don't permanently corrupt the base template
        current_cfg = {}
        for key, value in base_cfg.items():
            if isinstance(value, dict) and not key.startswith('__'):
                current_cfg[key] = copy.deepcopy(value)
                
        # Extract the current ne_points list
        ne_points = current_cfg['PLASMA']['ne_points']
        
        # Reconstruct the first tuple (edge density at x=0.0) with the new ne0
        current_ne_point = current_cfg['PLASMA']['ne_points'] = [(ne_points[0][0], ne0)] + ne_points[1:]

        print(f'current_ne_points: {current_ne_point}')
        try:
            # 2. Instantiate and run the solver pipeline
            solver = LHCouplingSolver_2DHcurl_1DH1(current_cfg, geom_mode, box_medium, antenna_grill)
            solver.build_mesh_with_PMLs()
            solver.build_physics_Stix_B_field()
            solver.solve_helmholtz_2DHcurl_1DH1_with_pml(solver.mesh, geom_mode, box_medium)
            
            # 3. Extract Waveguide S-Parameters
            gamma_dict = solver.compute_waveguide_S_parameters()
            
            # 4. Average the power reflection coefficients
            refl_list = [data['Power_Reflectivity'] for wg, data in gamma_dict.items()]
            
            if refl_list:
                avg_refl = np.mean(refl_list)
                avg_reflections[i] = avg_refl
                print(f">>> Success: Average Reflection = {avg_refl * 100:.2f}%")
            else:
                print(">>> Warning: No active waveguides found in gamma_dict.")
                avg_reflections[i] = np.nan
                
        except Exception as e:
            print(f">>> [!] SOLVER FAILED for ne0 = {ne0:.2e}. Error: {e}")
            avg_reflections[i] = np.nan # Log as NaN to keep array shapes consistent
            
    # 5. Save results robustly to disk
    np.savez(save_filename, ne0=ne0_values, reflection=avg_reflections)
    print(f"\n--- Scan Complete! Data saved to {save_filename} ---")
    
    return ne0_values, avg_reflections
ne0_values = np.array([1.2, 2, 3, 4, 5, 8, 10, 20, 30, 40])*1e17
# ne0_values = np.array([1.2, 10, 40])*1e17
#ne0_values, avg_reflections = run_ne0_parameter_scan(cfg.__dict__, ne0_values, geom_mode, box_medium, antenna_grill)

In [10]:
batch_mat_file = 'matlab_data_files/saved_batch_8mods_1wg.mat'
print(f"--- Structure file : {batch_mat_file} ---")
with h5py.File(batch_mat_file, 'r') as f:
    f.visititems(afficher_structure)

import numpy as np
import h5py
import matplotlib.pyplot as plt
# 2. Initialize Data Containers
ne0_values_aloha = []
avg_reflection_aloha = []

# Number of density steps in your scan
num_steps = 12 

print("--- Extracting Data from ALOHA #refs# Heap ---")

# 3. Open HDF5 File
with h5py.File(batch_mat_file, 'r') as h5f:
    refs_group = h5f['#refs#']
    
    for i in range(num_steps):
        # ASCII mapping: 'A' is 65, 'M' is 77
        folder_ne0 = chr(65 + i)      # A, B, C... L
        folder_coeff = chr(77 + i)    # M, N, O... X
        
        try:
            # Extract ne0
            # MATLAB scalars in h5py are 2D arrays of shape (1, 1)
            ne0_val = refs_group[folder_ne0]['ne0'][0, 0]
            
            # Extract CoeffRefPuiss
            # Shape is (1, 8), we extract the full array and compute the mean
            coeff_array = refs_group[folder_coeff]['CoeffRefPuiss'][:]
            avg_refl = np.mean(coeff_array)
            ne0_values_aloha.append(ne0_val)
            avg_reflection_aloha.append(avg_refl)
            
            print(f"Step {i+1:02d} | ne0 = {ne0_val:.2e} | Mean Reflection = {avg_refl:.4f}")
            
        except KeyError as e:
            print(f"[ERROR] Could not find expected paths for step {i+1}. Missing key: {e}")
            print(f"Looked in: #refs#/{folder_ne0}/ne0 and #refs#/{folder_coeff}/CoeffRefPuiss")

# Convert to numpy arrays
ne0_values_aloha = np.array(ne0_values_aloha)
avg_reflection_aloha = np.array(avg_reflection_aloha)

# 4. Rigorous Sorting
# Ensure the arrays are strictly ordered by ascending density
# This protects against MATLAB #refs# ID reuse or out-of-order execution
sort_indices = np.argsort(ne0_values_aloha)
ne0_aloha_sorted = ne0_values_aloha[sort_indices]
refl_aloha_sorted = avg_reflection_aloha[sort_indices]


try:
    fem_data = np.load("fem_scan_results.npz")
    ne0_fem = fem_data['ne0']
    refl_fem = fem_data['reflection']
    fem_available = True
except FileNotFoundError:
    print("FEM results not found. Plotting ALOHA only.")
    fem_available = False

# 5. Plotting
fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(ne0_aloha_sorted, refl_aloha_sorted, marker='o', markersize=8, 
    linestyle='-', color='royalblue', linewidth=2, label='ALOHA')
print(f'refl_fem: {refl_fem}')
if fem_available:
    ax.plot(ne0_fem, refl_fem*100, marker='o', markersize=8, 
            linestyle='--', color='crimson', linewidth=2, label='FEM')
    
ax.set_xlabel("Facing Edge Density ne0 [m-3]", fontsize=16)
ax.set_ylabel("Average Power Reflection [%]", fontsize=16)
# ax.set_title("ALOHA: Mean Reflection Coefficient vs Edge Density", fontsize=15, pad=15)

ax.grid(True, which='major', linestyle='-', alpha=0.6)
ax.grid(True, which='minor', linestyle=':', alpha=0.3)
ax.tick_params(direction='in', length=6, width=1.5, bottom=True, top=True, left=True, right=True)

ax.legend(loc='best', fontsize=12)

# If your density scan covers multiple orders of magnitude, a log scale is standard:
# ax.set_xscale('log')

plt.tight_layout()
#plt.show()

--- Structure file : matlab_data_files/saved_batch_8mods_1wg.mat ---


NameError: name 'h5py' is not defined

In [ ]:
# --- Extracting Data from ALOHA #refs# Heap ---
# Step 01 | ne0 = 1.20e+17 | Mean Reflection = 33.0896
# Step 02 | ne0 = 2.00e+17 | Mean Reflection = 16.2203
# Step 03 | ne0 = 3.00e+17 | Mean Reflection = 8.0000
# Step 04 | ne0 = 4.00e+17 | Mean Reflection = 4.4628
# Step 05 | ne0 = 5.00e+17 | Mean Reflection = 2.5916
# Step 06 | ne0 = 6.00e+17 | Mean Reflection = 1.6050
# Step 07 | ne0 = 8.00e+17 | Mean Reflection = 0.6842
# Step 08 | ne0 = 1.00e+18 | Mean Reflection = 0.6223
# Step 09 | ne0 = 2.00e+18 | Mean Reflection = 3.4875
# Step 10 | ne0 = 3.00e+18 | Mean Reflection = 7.2640
# Step 11 | ne0 = 4.00e+18 | Mean Reflection = 10.8247